# Type-I search performance benchmark for Sec. 5

This notebook benchmarks the **graph-guided Type-I cage search** against a conventional **full-spectrum dense exact diagonalization (ED)** baseline in the same microscopic symmetry sector.

The comparison is intentionally separated into:

1. common basis/Hamiltonian construction;
2. the incremental Type-I candidate/nullspace solve;
3. the incremental full dense diagonalization;
4. end-to-end peak resident memory measured in isolated subprocesses;
5. correctness/recall checks on tractable reference systems.

The ED baseline retains all eigenvectors, because an eigenstate-search workflow needs eigenvectors rather than eigenvalues alone. The Type-I method is not claimed to find every possible interference-caged state; this benchmark measures the restricted Type-I search defined in the draft.

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import os
import platform
import subprocess
import sys
import tempfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.linalg as la

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "qlinks").is_dir():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate the qlinks repository root.")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from helpers import save_prx_figure, set_revtex_matplotlib_style
from qlinks.basis.configs import basis_configs_from_build_result
from qlinks.caging import CageSearchConfig, CageSearcher
from qlinks.models import (
    SpinOneXYChainModel,
    SquareQDMModel,
    spin_one_xy_scar_tower_states,
)

DATA_DIR = REPO_ROOT / "experimental" / "data" / "type1_search_benchmark"
FIGURE_DIR = DATA_DIR / "figures"
RUNNER = REPO_ROOT / "experimental" / "benchmarks" / "type1_search_benchmark_runner.py"
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

TOL = 1.0e-10
USE_TEX = False
FIGURE_FORMATS = ("pdf", "svg")
BENCHMARK_REPEATS = 1          # use 3 or more for production timing
MAX_DENSE_ED_DIM = 2_000       # safety guard; raise on the remote server as memory allows
VALIDATION_MAX_DIM = 1_000     # correctness check, not part of timed benchmark

# Small defaults are deliberate so the notebook remains a smoke test in CI/sandbox.
# Suggested remote-server extension:
#   SPIN_LENGTHS = (6, 8, 10)
#   QDM_SIZES = ((4, 2), (4, 4), (6, 4))
SPIN_LENGTHS = (6, 8)
QDM_SIZES = ((4, 2), (4, 4), (6, 4))

set_revtex_matplotlib_style(base_font_size=8, prefer_tex=USE_TEX)
print("repo:", REPO_ROOT)
print("runner:", RUNNER)


## 1. Matched benchmark cases

For the spin-1 chain we work in the fixed magnetization sector $M=-2$ and include $D\sum_r(S_r^z)^2$ with $D=1$, so the known staggered tower lies in a nontrivial uniform-potential shell. For the square QDM we use periodic boundary conditions and the zero-winding sector with $\lambda=1$.

Both methods receive exactly the same already-defined microscopic sector. The runtime table records the common basis/Hamiltonian build separately from the method-specific solve.

### Why the spin-1 Type-I benchmark uses finite $D$

The spin-1 benchmark intentionally uses a nonzero single-ion anisotropy.  In a fixed-$M$ sector with $D=0$, the diagonal potential $hM$ is constant across the entire sector, so the uniform-potential label does not reduce the chiral candidate space: generic chiral zero modes of $K$ can mix with the target Type-I shell.  At finite $D$, the tower support is selected by
\[
\sum_r (S_r^z)^2=L,
\]
whereas configurations containing $|0\rangle$ have a smaller value.  Finite $D$ therefore makes the chiral-shell restriction operationally meaningful while leaving the target staggered tower exact.

In [ ]:
benchmark_cases = [
    {
        "label": f"spin-1 XY L={L}",
        "model": "spin1_xy",
        "L": int(L),
        "total_sz": -2,
        "j_xy": 1.0,
        "d_z": 1.0,
        "h_z": 0.0,
    }
    for L in SPIN_LENGTHS
]
benchmark_cases += [
    {
        "label": f"QDM {lx}x{ly}",
        "model": "square_qdm",
        "size": [int(lx), int(ly)],
        "winding_x": 0,
        "winding_y": 0,
        "coup_kin": -1.0,
        "coup_pot": 1.0,
    }
    for lx, ly in QDM_SIZES
]

pd.DataFrame(benchmark_cases)


## 2. Isolated runtime and peak-memory measurements

Each method is executed in a fresh Python subprocess. This makes the reported peak resident set size (RSS) meaningful: the Type-I and ED runs do not inherit one another's large arrays. The reported **method time** excludes the common basis/Hamiltonian construction, while **total time** includes it.

The Type-I run also records the candidate-shell dimensions, active-boundary shapes, boundary ranks/nullities, solver-stage timings, residuals, and the actual cache size of the candidate matrices. The ED run retains the complete eigenvector matrix.

In [ ]:
def run_isolated(case: dict, method: str, repeat_index: int) -> dict:
    output = DATA_DIR / f"_tmp_{case['label'].replace(' ', '_').replace('=', '').replace('x', 'x')}_{method}_{repeat_index}.json"
    command = [
        sys.executable,
        str(RUNNER),
        "--method", method,
        "--case-json", json.dumps(case),
        "--output", str(output),
        "--tolerance", str(TOL),
    ]
    subprocess.run(command, cwd=REPO_ROOT, check=True)
    result = json.loads(output.read_text())
    output.unlink(missing_ok=True)
    result["repeat_index"] = int(repeat_index)
    return result


raw_results = []
for case in benchmark_cases:
    type1_runs = []
    for repeat in range(BENCHMARK_REPEATS):
        row = run_isolated(case, "type1", repeat)
        raw_results.append(row)
        type1_runs.append(row)

    dimension = int(type1_runs[0]["hilbert_dimension"])
    if dimension <= MAX_DENSE_ED_DIM:
        for repeat in range(BENCHMARK_REPEATS):
            raw_results.append(run_isolated(case, "dense_ed", repeat))
    else:
        print(f"Skipping dense ED for {case['label']}: dimension {dimension} > {MAX_DENSE_ED_DIM}")

raw_df = pd.json_normalize(raw_results)
raw_df.to_csv(DATA_DIR / "type1_search_benchmark_raw.csv", index=False)
raw_df[[
    "case_label", "method", "hilbert_dimension", "build_seconds",
    "method_seconds", "total_seconds", "peak_rss_mb"
]]


## 3. Median performance table and algorithmic problem sizes

For production data, set `BENCHMARK_REPEATS >= 3`; the figure uses the median over repeats. Peak RSS is an end-to-end process measurement. Two deterministic storage diagnostics are also retained:

- dense ED lower bound: one dense Hamiltonian plus one dense eigenvector matrix;
- Type-I storage proxy: sparse $H,K,V$, cached candidate blocks, and a conservative dense workspace for the largest candidate shell.

The storage proxy is useful when small-system RSS is dominated by the Python/SciPy runtime itself.

In [ ]:
summary_rows = []
for (case_label, method), group in raw_df.groupby(["case_label", "method"], sort=False):
    first = group.iloc[0]
    row = {
        "case_label": case_label,
        "model": first["model"],
        "method": method,
        "hilbert_dimension": int(first["hilbert_dimension"]),
        "build_seconds": float(group["build_seconds"].median()),
        "method_seconds": float(group["method_seconds"].median()),
        "total_seconds": float(group["total_seconds"].median()),
        "peak_rss_mb": float(group["peak_rss_mb"].median()),
    }
    if method == "Type-I search":
        row.update({
            "n_type1_candidates": int(first["n_type1_candidates"]),
            "largest_candidate_dimension": int(first["largest_candidate_dimension"]),
            "candidate_fraction": float(first["largest_candidate_dimension"] / first["hilbert_dimension"]),
            "n_cage_records": int(first["n_cage_records"]),
            "max_full_residual": float(first["max_full_residual"]),
            "storage_proxy_mb": float(first["type1_storage_proxy_bytes"] / 2**20),
        })
    else:
        row.update({
            "n_type1_candidates": np.nan,
            "largest_candidate_dimension": np.nan,
            "candidate_fraction": np.nan,
            "n_cage_records": np.nan,
            "max_full_residual": np.nan,
            "storage_proxy_mb": float(first["dense_storage_lower_bound_bytes"] / 2**20),
        })
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(DATA_DIR / "type1_search_benchmark_summary.csv", index=False)
display(summary_df)

comparison_rows = []
for case_label, group in summary_df.groupby("case_label", sort=False):
    methods = group.set_index("method")
    if "Type-I search" not in methods.index or "Dense ED" not in methods.index:
        continue
    type1 = methods.loc["Type-I search"]
    ed = methods.loc["Dense ED"]
    comparison_rows.append({
        "case_label": case_label,
        "model": type1["model"],
        "hilbert_dimension": int(type1["hilbert_dimension"]),
        "runtime_speedup_ED_over_TypeI": float(ed["method_seconds"] / type1["method_seconds"]),
        "peak_rss_ratio_ED_over_TypeI": float(ed["peak_rss_mb"] / type1["peak_rss_mb"]),
        "storage_proxy_ratio_ED_over_TypeI": float(ed["storage_proxy_mb"] / type1["storage_proxy_mb"]),
        "largest_candidate_fraction": float(type1["candidate_fraction"]),
    })
comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(DATA_DIR / "type1_search_benchmark_comparison.csv", index=False)
display(comparison_df)


## 4. Draft-ready PRX benchmark figure

Panel (a) compares the **incremental solver time** after the microscopic sector has been built. Panel (b) compares the isolated-process peak RSS. Model labels are annotated so that the points are not mistaken for a single universal scaling curve across different Hilbert-space constraints.

For the manuscript benchmark, extend each model along its own size sequence on the remote server and keep the raw CSV files with the hardware/software metadata below.

In [ ]:
fig = plt.figure(figsize=(7.0, 2.75))
grid = fig.add_gridspec(1, 2, wspace=0.34)
ax_runtime = fig.add_subplot(grid[0, 0])
ax_memory = fig.add_subplot(grid[0, 1])

model_markers = {"spin1_xy": "o", "square_qdm": "s"}
method_linestyles = {"Type-I search": "-", "Dense ED": "--"}

# Connect points only within the same model family. The two methods are
# distinguished by line style; model families by marker shape.
for model in summary_df["model"].unique():
    for method in ("Type-I search", "Dense ED"):
        subset = summary_df[(summary_df["model"] == model) & (summary_df["method"] == method)].sort_values("hilbert_dimension")
        if subset.empty:
            continue
        label = f"{method}; " + ("spin-1 XY" if model == "spin1_xy" else "square QDM")
        ax_runtime.loglog(
            subset["hilbert_dimension"], subset["method_seconds"],
            marker=model_markers[model], linestyle=method_linestyles[method], label=label,
        )
        ax_memory.loglog(
            subset["hilbert_dimension"], subset["peak_rss_mb"],
            marker=model_markers[model], linestyle=method_linestyles[method], label=label,
        )

for _, row in summary_df.iterrows():
    short = row["case_label"].replace("spin-1 XY ", "XY ")
    ax_runtime.annotate(short, (row["hilbert_dimension"], row["method_seconds"]), xytext=(3, 3), textcoords="offset points", fontsize=6)
    ax_memory.annotate(short, (row["hilbert_dimension"], row["peak_rss_mb"]), xytext=(3, 3), textcoords="offset points", fontsize=6)

ax_runtime.set_xlabel(r"Physical-sector dimension $\mathcal{D}$")
ax_runtime.set_ylabel("Method time (s)")
ax_runtime.grid(alpha=0.25)
ax_runtime.legend(frameon=False, fontsize=6)
ax_runtime.text(0.02, 0.98, "(a)", transform=ax_runtime.transAxes, ha="left", va="top")

ax_memory.set_xlabel(r"Physical-sector dimension $\mathcal{D}$")
ax_memory.set_ylabel("Peak RSS (MB)")
ax_memory.grid(alpha=0.25)
ax_memory.legend(frameon=False, fontsize=6)
ax_memory.text(0.02, 0.98, "(b)", transform=ax_memory.transAxes, ha="left", va="top")

save_prx_figure(
    fig, "type1_search_benchmark", directory=FIGURE_DIR, formats=FIGURE_FORMATS,
)
plt.show()

# Deterministic storage-proxy companion, useful when small-run RSS is import dominated.
fig, ax = plt.subplots(figsize=(3.35, 2.45))
for model in summary_df["model"].unique():
    for method in ("Type-I search", "Dense ED"):
        subset = summary_df[(summary_df["model"] == model) & (summary_df["method"] == method)].sort_values("hilbert_dimension")
        if subset.empty:
            continue
        label = f"{method}; " + ("XY" if model == "spin1_xy" else "QDM")
        ax.loglog(
            subset["hilbert_dimension"], subset["storage_proxy_mb"],
            marker=model_markers[model], linestyle=method_linestyles[method], label=label,
        )
ax.set_xlabel(r"Physical-sector dimension $\mathcal{D}$")
ax.set_ylabel("Matrix-storage proxy (MB)")
ax.grid(alpha=0.25)
ax.legend(frameon=False, fontsize=6)
save_prx_figure(fig, "type1_search_storage_proxy", directory=FIGURE_DIR, formats=FIGURE_FORMATS)
plt.show()


## 5. Type-I stage breakdown and boundary problems

This table supplies the reproducibility details requested by the draft note: candidate shell, boundary shape, rank/nullity, solver stages, tolerance, and finite-size residual. It also makes clear where the graph-guided runtime is spent.

In [ ]:
stage_rows = []
candidate_rows = []
for result in raw_results:
    if result["method"] != "Type-I search" or result["repeat_index"] != 0:
        continue
    for stage, seconds in result["search_stage_seconds"].items():
        stage_rows.append({
            "case_label": result["case_label"],
            "hilbert_dimension": result["hilbert_dimension"],
            "stage": stage,
            "seconds": seconds,
        })
    for candidate in result["candidate_summaries"]:
        candidate_rows.append({
            "case_label": result["case_label"],
            "hilbert_dimension": result["hilbert_dimension"],
            **candidate,
        })

stage_df = pd.DataFrame(stage_rows)
candidate_df = pd.DataFrame(candidate_rows)
stage_df.to_csv(DATA_DIR / "type1_search_stage_timings.csv", index=False)
candidate_df.to_csv(DATA_DIR / "type1_candidate_boundary_problems.csv", index=False)
display(candidate_df.head(20))

fig, ax = plt.subplots(figsize=(3.35, 2.55))
major_stages = ["candidate_build_type1", "solve_type1", "rank_deduplication"]
pivot = stage_df[stage_df["stage"].isin(major_stages)].pivot(index="case_label", columns="stage", values="seconds").fillna(0.0)
pivot.plot(kind="bar", stacked=True, ax=ax, legend=True)
ax.set_xlabel("")
ax.set_ylabel("Type-I search time (s)")
ax.tick_params(axis="x", rotation=30)
ax.legend(frameon=False, fontsize=6)
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
save_prx_figure(fig, "type1_search_stage_breakdown", directory=FIGURE_DIR, formats=FIGURE_FORMATS)
plt.show()


## 6. Correctness and recall checks on tractable cases

These checks are deliberately outside the timed benchmark.

- Every Type-I record is projected onto the complete ED eigenspace at the same energy. The basis-independent projector weight should be one even when ED returns arbitrary vectors inside a degenerate multiplet.
- For the spin-1 chain, the analytically known staggered tower state at $M=-2$ must lie in the Type-I $(\kappa=0,v=L)$ subspace.
- For the $4\times4$ zero-winding QDM, the search must reproduce the known Type-I counts $(0,4):9$ and $(0,6):1$.

This is the appropriate notion of recall in the presence of exact degeneracies; comparing individual ED eigenvectors would be basis dependent.

In [ ]:
def build_case_in_process(case):
    if case["model"] == "spin1_xy":
        model = SpinOneXYChainModel(
            length=int(case["L"]),
            boundary_condition="periodic",
            j_xy=float(case["j_xy"]),
            d_z=float(case["d_z"]),
            h_z=float(case["h_z"]),
            total_sz=int(case["total_sz"]),
        )
        return model.build(builder="optimized", basis_solver="dfs", sort_basis=True)
    lx, ly = case["size"]
    model = SquareQDMModel(
        lx=int(lx), ly=int(ly), boundary_condition="periodic",
        winding_x=0, winding_y=0, coup_kin=-1.0, coup_pot=1.0,
    )
    return model.build(builder="sparse", backend="scipy", basis_solver="dfs", sort_basis=True)


validation_rows = []
for case in benchmark_cases:
    dimension_row = summary_df[(summary_df["case_label"] == case["label"]) & (summary_df["method"] == "Type-I search")].iloc[0]
    if int(dimension_row["hilbert_dimension"]) > VALIDATION_MAX_DIM:
        continue

    build = build_case_in_process(case)
    search = CageSearcher.from_model_build_result(
        build,
        config=CageSearchConfig(
            search_type="type1",
            tolerance=TOL,
            validate_full_residual=True,
            degenerate_basis_strategy="none",
            store_full_states=False,
        ),
    ).run()
    evals, evecs = la.eigh(build.hamiltonian.toarray(), check_finite=False)

    ed_weights = []
    for record in search.records:
        psi = np.zeros(build.hamiltonian.shape[0], dtype=np.complex128)
        psi[np.asarray(record.cage_state.support, dtype=np.int64)] = record.cage_state.local_state
        energy = float(np.real(record.cage_state.energy))
        mask = np.abs(evals - energy) <= 1.0e-8
        weight = float(np.sum(np.abs(evecs[:, mask].conj().T @ psi) ** 2))
        ed_weights.append(weight)

    analytic_recall = np.nan
    if case["model"] == "spin1_xy":
        configs = basis_configs_from_build_result(build)
        tower_states, _labels = spin_one_xy_scar_tower_states(
            basis_configs=configs,
            length=int(case["L"]),
            normalize=True,
        )
        if tower_states.shape[1] != 1:
            raise RuntimeError("Expected exactly one tower state in the fixed-M basis.")
        target = tower_states[:, 0]
        signature = (0, int(case["L"]))
        found = search.full_state_matrix(signature)
        projector_weight = float(np.sum(np.abs(found.conj() @ target) ** 2)) if len(found) else 0.0
        analytic_recall = projector_weight
        if projector_weight < 1.0 - 1.0e-8:
            raise AssertionError(f"Spin-1 tower recall failed for {case['label']}: {projector_weight}")

    if case["model"] == "square_qdm" and tuple(case["size"]) == (4, 4):
        expected = {(0, 4): 9, (0, 6): 1}
        for signature, count in expected.items():
            if search.counts_by_signature.get(signature, 0) != count:
                raise AssertionError(
                    f"QDM 4x4 count mismatch for {signature}: "
                    f"{search.counts_by_signature.get(signature, 0)} != {count}"
                )
        analytic_recall = 1.0

    min_ed_weight = min(ed_weights, default=1.0)
    if min_ed_weight < 1.0 - 1.0e-8:
        raise AssertionError(f"ED eigenspace containment failed for {case['label']}: {min_ed_weight}")

    validation_rows.append({
        "case_label": case["label"],
        "hilbert_dimension": build.hamiltonian.shape[0],
        "n_type1_records": len(search.records),
        "min_ED_eigenspace_projector_weight": min_ed_weight,
        "known_state_or_count_recall": analytic_recall,
        "max_full_residual": max(
            (float(record.cage_state.full_residual or 0.0) for record in search.records),
            default=0.0,
        ),
    })

validation_df = pd.DataFrame(validation_rows)
validation_df.to_csv(DATA_DIR / "type1_search_recall_validation.csv", index=False)
display(validation_df)


## 7. Hardware/software manifest and interpretation

The benchmark figure should only be moved into the manuscript together with this manifest. Runtime numbers are hardware dependent; the scientifically robust quantities are the matched physical sector, solver definition, candidate reduction, residual/recall checks, and the trend of the comparison on a fixed machine.

For a stronger production benchmark on the remote server:

```python
BENCHMARK_REPEATS = 3
SPIN_LENGTHS = (6, 8, 10)
QDM_SIZES = ((4, 2), (4, 4), (6, 4))
MAX_DENSE_ED_DIM = 10_000  # only after checking available RAM
```

Do not extend dense ED to a size whose $2\mathcal D^2$ complex-array lower bound is unsafe for the machine.

In [ ]:
def git_revision():
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"], cwd=REPO_ROOT, text=True, stderr=subprocess.DEVNULL
        ).strip()
    except Exception:
        return "unavailable"

manifest = pd.DataFrame([{
    "git_revision": git_revision(),
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "processor": platform.processor(),
    "logical_cpu_count": os.cpu_count(),
    "tolerance": TOL,
    "benchmark_repeats": BENCHMARK_REPEATS,
    "max_dense_ed_dim": MAX_DENSE_ED_DIM,
    "type1_solver": "CageSearcher(search_type='type1', degenerate_basis_strategy='none')",
    "ed_solver": "scipy.linalg.eigh(full dense H, eigenvectors retained)",
    "timing_definition": "method time excludes common model/basis build",
    "memory_definition": "isolated-process peak RSS; deterministic storage proxies also recorded",
}])
manifest.to_csv(DATA_DIR / "type1_search_benchmark_manifest.csv", index=False)
display(manifest)

print("Generated files:")
for path in sorted(DATA_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(REPO_ROOT))
